## European Cities 2D [Part 1]

In [ ]:
import numpy as np
import torch.nn as nn
import torch
import matplotlib.pyplot as plt
import torch.optim as optim

paris_coords = np.array([
    [48.8575, 2.3514], #Center of Paris
    [48.8584, 2.2945], # Eiffel Tower
    [48.8530, 2.3499], #Notre Dame
    [48.8606,  2.3376], #Louvre
    [48.8606, 2.3522]  #Centre Pompidou
])

madrid_coords = np.array([
    [40.4167, -3.7033],   # Center of Madrid
    [40.4153, -3.6835],   # Retiro Park 
    [40.4180, -3.7143],   # Royal Palace 
    [40.4138, -3.6921],   # Prado Museum 
    [40.4169, -3.7033]   # Puerta del Sol 
])

berlin_coords = np.array([
    [52.5200, 13.4050], # Center of Berlin
    [52.5163, 13.3777],   # Brandenburg Gate 
    [52.5169, 13.4019],   # Museum Island 
    [52.5074, 13.3904],   # Checkpoint Charlie 
    [52.5251, 13.3694]   # Berlin Central Station 
])

barcelona_coords = np.array([
    [41.3874, 2.1686],    # Center of Barcelona
    [41.4036, 2.1744],    # Sagrada Familia
    [41.3819, 2.1773],    # Gothic Quarter 
    [41.4145, 2.1527],    # Park Güell 
    [41.3809, 2.1228],    # Camp Nou
])


In [ ]:
mean_to_subtract=np.array([48.8575, 2.3514]) #Center of Paris

In [ ]:
# [28, 2.2]

In [ ]:
# Set random seed for reproducibility
random_seed=25
torch.manual_seed(random_seed)
np.random.seed(random_seed)

In [ ]:
# Combine data into one matrix X and labels y 
X_raw = np.vstack([madrid_coords, paris_coords, berlin_coords, barcelona_coords]) #, brussels_coords, vienna_coords])
y = np.array([0, 0, 0, 0, 0,  #  Madrid labels (0)
              1, 1, 1, 1, 1,  #  Paris labels (1)
              2, 2, 2, 2, 2, # Berlin labels (2)
              3, 3, 3, 3, 3]) # Barcelona Labels 
              #3, 3, 3, 3, 3, 3, 3, 
              #4, 4, 4, 4, 4, 4, 4]) 

# Let's use only longitude as input for the simple model
# X_lon = X_raw[:, 1].reshape(-1, 1)  # Extract longitude and reshape to column vector

# Normalize data (simple scaling by dividing by 100, I guess I could do 10)
X = X_raw # / 100 - Simple longitude only problem does not seem to require normalization, that's interesting!
X = X_raw - mean_to_subtract #np.array([28, 2.2]) #Ok looks like I might need to normalize liek this for this one to converge

# Alternative normalization (uncomment to use)
# mean = np.mean(X_lon)
# std = np.std(X_lon)
# X = (X_lon - mean) / std

rI=np.arange(len(y))
np.random.shuffle(rI)
X=X[rI,:]
y=y[rI]
X_raw=X_raw[rI,:]

print(f"Shape of X: {X.shape}")
print(f"Shape of y: {y.shape}")

In [ ]:
X.max(0)

In [ ]:
X.min(0)

xs.shape

In [ ]:
class TinyGPSModel(nn.Module):
    def __init__(self, input_size=2, output_size=4):
        super(TinyGPSModel, self).__init__()
        self.output = nn.Linear(input_size, output_size) #, bias=False)  # 3 cities

    def forward(self, x):
        x = self.output(x)
        return x

In [ ]:
# Initialize model, loss, and optimizer
model = TinyGPSModel()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

#I can manually initialize and still learns real good? yeah seems like it!
# with torch.no_grad():
#     model.output.weight[0,0]=1.0
#     model.output.weight[1,0]=0.0
#     model.output.weight[2,0]=-1.0
#     model.output.bias[0]=0
#     model.output.bias[1]=0
#     model.output.bias[2]=0

In [ ]:
#Where to save training heatmaps
import os
save_dir='data/backprop2/graphics/to_manim/jun_6_2'
os.makedirs(save_dir, exist_ok=True)

In [ ]:
#Swaggin - need to sync with Sam on exact numbers
# min_long=-6.0
# max_long=17.0
# min_lat=36.0
# max_lat=56.0
min_long=-6.0
max_long=16.5
min_lat=38.6
max_lat=53.5  
num_steps=256
heatmap_viz_logit_multiplier=8 #Makes things more winner take all for cleaner logit viz
heatmaps=[np.zeros((num_steps, num_steps)) for i in range(8)]

for i, lat in enumerate(np.linspace(max_lat, min_lat, num_steps)):
    for j, long in enumerate(np.linspace(min_long, max_long, num_steps)):
        with torch.no_grad():
            logits=model(torch.tensor([lat, long], dtype=torch.float)).detach()
            yhat=torch.nn.Softmax(0)(heatmap_viz_logit_multiplier*logits).numpy()

        for k in range(4):
            heatmaps[k][i,j]=logits.numpy()[k]
            heatmaps[k+3][i,j]=yhat[k]

In [ ]:
min_lat-mean_to_subtract[0], max_lat-mean_to_subtract[0]

In [ ]:
min_long-mean_to_subtract[1], max_long-mean_to_subtract[1]

In [ ]:
#-13, 6, -8, 15

In [ ]:
import matplotlib.colors as mcolors
def create_transparent_colormap(color='cyan', name='transparent_to_color'):
    # Convert color name to RGBA
    base_color = mcolors.to_rgba(color)
    
    # Create colormap: transparent (alpha=0) to full color (alpha=1)
    colors = [(base_color[0], base_color[1], base_color[2], 0),  # transparent
              (base_color[0], base_color[1], base_color[2], 1)]  # full color
    
    n_bins = 256
    cmap = mcolors.LinearSegmentedColormap.from_list(name, colors, N=n_bins)
    return cmap

# Create the colormap
transparent_cyan_cmap = create_transparent_colormap('cyan')
transparent_yellow_cmap = create_transparent_colormap('#ffd35a')
transparent_green_cmap = create_transparent_colormap('#00a14b')
transparent_magenta_cmap = create_transparent_colormap('magenta')

In [ ]:
cmaps=[transparent_cyan_cmap, transparent_yellow_cmap, transparent_green_cmap, transparent_magenta_cmap, 
       transparent_cyan_cmap, transparent_yellow_cmap, transparent_green_cmap, transparent_magenta_cmap]
save_names=['_logits_1.png', '_logits_2.png', '_logits_3.png', '_logits_4.png', 
            '_yhat_1.png', '_yhat_2.png', '_yhat_3.png', '_yhat_4.png']

# for l in range(6):
#     plt.clf()
#     plt.figure(frameon=False)
#     ax = plt.Axes(plt.gcf(), [0., 0., 1., 1.])
#     ax.set_axis_off()
#     plt.gcf().add_axes(ax)
#     plt.imshow(heatmaps[l],  cmap=cmaps[l]) #np.rot90(heatmaps[0])) #Wait and see if I need to rotate or transpose
#     plt.savefig(save_dir+save_names[l], bbox_inches='tight', pad_inches=0, dpi=300)
#     plt.close()

In [ ]:
# Initialize model, loss, and optimizer
model = TinyGPSModel()
criterion = nn.CrossEntropyLoss()
# optimizer = optim.SGD(model.parameters(), lr=0.5)
optimizer = optim.Adam(model.parameters(), lr=0.1)

#I can manually initialize and still learns real good? yeah seems like it!
# with torch.no_grad():
#     model.output.weight[0,0]=1.0
#     model.output.weight[1,0]=0.0
#     model.output.weight[2,0]=-1.0
#     model.output.bias[0]=0
#     model.output.bias[1]=0
#     model.output.bias[2]=0

weights=[]
grads=[]
xs=[]
ys=[]
logitss=[]
yhats=[]

# Training loop
for i in range(350):
    xs.append(X_raw[i%len(y)])
    ys.append(y[i%len(y)])
    weights.append(np.concatenate([model.output.weight.detach().numpy().ravel(), model.output.bias.detach().numpy().ravel()]))
    
    optimizer.zero_grad()
    
    #Stochastic - i think this is a better starting point pedagogically. 
    outputs = model(torch.tensor(X[i%len(y)]).float())
    loss = criterion(outputs, torch.tensor(y[i%len(y)])) 

    logitss.append(outputs.detach().numpy())
    yhats.append(torch.nn.Softmax(0)(outputs.detach()).numpy())
    
    loss.backward()  # backpropagation
    grads.append(np.concatenate([model.output.weight.grad.detach().numpy().ravel(), model.output.bias.grad.detach().numpy().ravel()]))
    optimizer.step() #
    
    if (i + 1) % 10 == 0:
        with torch.no_grad():
            logits=model(torch.tensor(X, dtype=torch.float)) 
            accuracy=(torch.argmax(logits, dim=1)==torch.tensor(y)).sum().item()/len(y)
        print(f"Step {i+1}/{i}, Loss: {loss.item():.4f}, 'Accuracy: {accuracy:.4f}")

weights=np.array(weights)
grads=np.array(grads)
xs=np.array(xs)
ys=np.array(ys)
logitss=np.array(logitss)
yhats=np.array(yhats)

In [ ]:
# Initialize model, loss, and optimizer
model = TinyGPSModel()
criterion = nn.CrossEntropyLoss()
# optimizer = optim.SGD(model.parameters(), lr=0.01)
optimizer = optim.Adam(model.parameters(), lr=0.03)

#I can manually initialize and still learns real good? yeah seems like it!
# with torch.no_grad():
#     model.output.weight[0,0]=1.0
#     model.output.weight[1,0]=0.0
#     model.output.weight[2,0]=-1.0
#     model.output.bias[0]=0
#     model.output.bias[1]=0
#     model.output.bias[2]=0

weights=[]
grads=[]
xs=[]
ys=[]
logitss=[]
yhats=[]

# Training loop
for i in range(500):
    xs.append(X_raw[i%len(y)])
    ys.append(y[i%len(y)])
    weights.append(np.concatenate([model.output.weight.detach().numpy().ravel(), model.output.bias.detach().numpy().ravel()]))
    
    optimizer.zero_grad()
    
    #Stochastic - i think this is a better starting point pedagogically. 
    outputs = model(torch.tensor(X[i%len(y)]).float())
    loss = criterion(outputs, torch.tensor(y[i%len(y)])) 

    logitss.append(outputs.detach().numpy())
    yhats.append(torch.nn.Softmax(0)(outputs.detach()).numpy())

    #Heatmaps
    heatmaps=[np.zeros((num_steps, num_steps)) for i in range(8)]
    for j, lat in enumerate(np.linspace(max_lat, min_lat, num_steps)):
        for k, long in enumerate(np.linspace(min_long, max_long, num_steps)):
            with torch.no_grad():
                coords_norm=np.array([lat, long])-mean_to_subtract
                logits=model(torch.tensor(coords_norm.ravel(), dtype=torch.float)).detach()
                yhat=torch.nn.Softmax(0)(heatmap_viz_logit_multiplier*logits).numpy()
    
            for l in range(4):
                heatmaps[l][j, k]=logits.numpy()[l]
                heatmaps[l+4][j,k]=yhat[l]

    for l in range(8):
        plt.clf()
        plt.figure(frameon=False)
        ax = plt.Axes(plt.gcf(), [0., 0., 1., 1.])
        ax.set_axis_off()
        plt.gcf().add_axes(ax)
        plt.imshow(heatmaps[l],  cmap=cmaps[l]) #np.rot90(heatmaps[0])) #Wait and see if I need to rotate or transpose
        plt.savefig(save_dir+'/'+str(i)+save_names[l], bbox_inches='tight', pad_inches=0, dpi=300)
        plt.close()
    
    loss.backward()  # backpropagation
    grads.append(np.concatenate([model.output.weight.grad.detach().numpy().ravel(), model.output.bias.grad.detach().numpy().ravel()]))
    optimizer.step() #
    
    if (i + 1) % 10 == 0:
        with torch.no_grad():
            logits=model(torch.tensor(X, dtype=torch.float)) 
            accuracy=(torch.argmax(logits, dim=1)==torch.tensor(y)).sum().item()/len(y)
        print(f"Step {i+1}/{i}, Loss: {loss.item():.4f}, 'Accuracy: {accuracy:.4f}")

weights=np.array(weights)
grads=np.array(grads)
xs=np.array(xs)
ys=np.array(ys)
logitss=np.array(logitss)
yhats=np.array(yhats)

In [ ]:
logits.numpy()

In [ ]:
all_training_data=np.hstack((xs, ys.reshape(-1, 1), weights, grads, logitss, yhats))
np.save('data/backprop2/hackin/cities_2d_4', all_training_data)

In [ ]:
all_training_data.shape

In [ ]:
xs.shape

In [ ]:
ys.shape

In [ ]:
weights.shape

In [ ]:
grads.shape

In [ ]:
logitss.shape

In [ ]:
yhats.shape

In [ ]:
model.output.weight.detach().numpy().ravel()

In [ ]:
model.output.weight.detach().numpy()